# Forward Prediction as Post-processing: Lenvatinib

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laboratoire-de-Chemoinformatique/SynPlanner/blob/main/tutorials/20_Forward_Prediction.ipynb)

Plan routes to **Lenvatinib**, save the completed search, and use the SynFwd-1
Chemformer model to check whether each reaction's reactants recover its expected
product. Then aggregate the reaction scores, inspect the routes, and export JSON
and an offline HTML report.

This extends [05 Retrosynthetic Planning](05_Retrosynthetic_Planning.ipynb).
Forward assessment runs **after planning** and keeps every assessed route,
including routes with failed predictions. It does not change the search.

In [ ]:
# Colab setup — this cell does nothing when you run the notebook locally.
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [
            "pip",
            "install",
            "-q",
            "git+https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner.git@main",
        ],
        check=True,
    )
    print("SynPlanner installed. If an import below fails, restart the runtime.")

## 1. Set up paths and the Chemformer bundle

Run locally from the repository root or `tutorials/`. In Colab, upload or mount a
prepared SynFwd-1 ONNX bundle first. The forward bundle is separate from the GPS
planning preset and is not included in the Git repository. It is not currently
published in the SynPlanner Hugging Face data repository.

The default `v2` bundle uses KV-cached decoding and contains `encoder.onnx`,
`decoder.onnx`, `decoder_with_past.onnx`, `vocabulary.json`, and `model.json`.
Keep its attribution files with it. Set `SYNPLAN_CHEMFORMER_DIR`, or edit
`model_dir` below, to use another location. The older `v1` two-graph bundle also
works when its directory is selected; it takes longer on CPU.

The notebook uses ONNX on CPU, with four inference threads. No Chemformer
checkout or PyTorch is required for this workflow. Model files, the JSON
prediction cache, and generated reports stay outside Git.

In [ ]:
import os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
data_dir = project_root / "synplan_data"
results_dir = project_root / "tutorials/tutorial_results/lenvatinib_forward"
results_dir.mkdir(parents=True, exist_ok=True)

model_dir = Path(
    os.environ.get("SYNPLAN_CHEMFORMER_DIR", data_dir / "forward/synfwd-1/v2")
).expanduser()
routes_path = results_dir / "lenvatinib_routes.json"
cache_path = results_dir / "forward_cache.json"

print(f"Model: {model_dir}")
print(f"Results: {results_dir}")

## 2. Prepare Lenvatinib

We use the supplied SMILES exactly as the input to SynPlanner's normal target
preparation. `clean_stereo=False` preserves any supplied stereochemistry; this
particular target has no explicit stereo labels.

Lenvatinib is present in the Mcule/MolPort catalogue. The low-level `Tree` API
used below starts with the target scheduled for expansion, so we can explore
synthesis routes while leaving the downloaded catalogue unchanged.

In [ ]:
from IPython.display import display

from synplan.chem.utils import mol_from_smiles

LENVATINIB_SMILES = "C4CC4NC(=O)Nc3ccc(cc3Cl)Oc1ccnc(cc2OC)c1cc2C(=O)N"
target = mol_from_smiles(LENVATINIB_SMILES, clean_stereo=False)
display(target)

## 3. Run ordinary planning and save its routes

Download only the GPS ONNX ranking policy, reaction rules, and Mcule/MolPort
stock needed for this example. These are the corresponding resources from the
`synplanner-gps-mcule-molport` preset. The first catalogue load creates a reusable
local index; later loads reuse it.

The search uses 100 iterations, a 600-second limit, depth 6, and rollout
evaluation. No forward assessor is attached to the tree. Search limits apply to
planning only; the later Chemformer assessment has its own elapsed time.

If `lenvatinib_routes.json` already exists, this section reuses it. Choose a new
results directory to run a fresh search after changing the target or settings.
If you already have a v1 route file, set `routes_path` to it and skip planning;
the assessment sections need only the saved `Route` objects.

In [ ]:
from synplan.utils.loading import download_selected_files

if not routes_path.exists():
    download_selected_files(
        [
            ("policy/supervised_gps/v1", "reaction_rules.tsv"),
            ("policy/supervised_gps/v1/v1", "ranking_policy.onnx"),
            ("building_blocks/mcule-molport-2026-08", "building_blocks.json.gz"),
        ],
        save_to=data_dir,
        extract_zips=False,
    )

In [ ]:
import random
from time import perf_counter

import numpy as np

from synplan.chem.reaction.routes.io import read_routes_json, write_routes_json
from synplan.mcts.config import RolloutEvaluationConfig, TreeConfig
from synplan.mcts.record import write_search_record
from synplan.mcts.tree import Tree
from synplan.utils.loading import (
    load_building_blocks,
    load_evaluation_function,
    load_policy_function,
    load_reaction_rules,
)

if not routes_path.exists():
    random.seed(0)
    np.random.seed(0)
    building_blocks = load_building_blocks(
        data_dir / "building_blocks/mcule-molport-2026-08/building_blocks.json.gz"
    )
    reaction_rules = load_reaction_rules(
        data_dir / "policy/supervised_gps/v1/reaction_rules.tsv"
    )
    policy = load_policy_function(
        weights_path=data_dir / "policy/supervised_gps/v1/v1/ranking_policy.onnx"
    )
    tree_config = TreeConfig(
        max_iterations=100, max_time=600, max_depth=6, silent=False
    )
    evaluation = load_evaluation_function(
        RolloutEvaluationConfig(
            policy_network=policy,
            reaction_rules=reaction_rules,
            building_blocks=building_blocks,
            min_mol_size=tree_config.min_mol_size,
            max_depth=tree_config.max_depth,
        )
    )
    tree = Tree(
        target=target,
        config=tree_config,
        reaction_rules=reaction_rules,
        building_blocks=building_blocks,
        expansion_function=policy,
        evaluation_function=evaluation,
    )
    started = perf_counter()
    tree.run()
    print(f"Planning wall time: {perf_counter() - started:.2f} s")
    write_search_record(tree, results_dir / "lenvatinib_search.json.gz")
    write_routes_json(tree.routes(), routes_path)

all_routes = read_routes_json(routes_path, as_routes=True)
print(f"Loaded {len(all_routes)} completed routes from {routes_path.name}")

The unassessed routes are saved **before loading Chemformer**. An interrupted
assessment therefore does not require another search. A bounded search is not
guaranteed to find a route; if none were found, keep the saved search for
inspection rather than treating an unfinished route as solved.

For a manageable CPU example that also demonstrates route aggregation, assess
the first ten routes with at least two steps, keeping their saved order. Set
`route_limit = None` to assess the entire collection, including single-step
routes. This selection uses only route length, never the forward score.

In [ ]:
route_limit = 10  # None assesses all saved routes, including single-step routes.
routes = (
    all_routes
    if route_limit is None
    else [route for route in all_routes if len(route) > 1][:route_limit]
)
if not routes:
    raise RuntimeError(
        "No routes selected. Set route_limit=None to include single-step routes, "
        "or inspect the saved search if the collection is empty."
    )
print(f"Assessing {len(routes)} of {len(all_routes)} routes")
print(f"Reaction occurrences: {sum(len(route) for route in routes)}")

## 4. Load Chemformer and assess the reactions

`ForwardPredictor` loads the encoder and decoder sessions once. The bundle
metadata selects cached decoding automatically. We keep the shared generation
policy: up to ten deterministic input augmentations, twenty beams per
augmentation, and up to ten distinct final products. `top_k=10` is the matching
cutoff, not the beam width.

The assessor takes **all reactants of each step**, including stock molecules
and small fragments. Atom mapping is removed through chython formatting, then
RDKit canonical isomeric SMILES are used to prepare the model inputs.
Stereochemistry and isotopes are retained. Input augmentations pass directly to
the tokenizer without being canonicalized again.

Invalid generated products are discarded. Chython canonicalizes products with
`fix_tautomers=False`; complete stereo-free InChIKeys collapse duplicate product
identities and match the designated `step.product`. Standard InChI normalization
still applies. Matching ignores stereochemistry but distinguishes isotopes.

In [ ]:
from synplan.ml.inference import ForwardPredictor

started = perf_counter()
model = ForwardPredictor(
    model_dir,
    model_type="chemformer",
    backend="onnx",
    device="cpu",
    threads=4,
)
print(f"Model initialization: {perf_counter() - started:.2f} s")

In [ ]:
from synplan.chem.reaction.routes.quality import ForwardReactionAssessor

assessor = ForwardReactionAssessor(model, cache_path=cache_path, batch_size=1, top_k=10)
annotated_routes = assessor.assess(routes, silent=False)
first_stats = dict(assessor.last_stats)

`batch_size=1` bounds CPU memory use and saves the cache after each completed
input; each input still receives the full augmentation and beam settings.
The two progress bars count routes being collected and uncached inputs being
predicted. Repeated reactants across routes reuse one prediction, even when
the expected products differ.

Each `Step.forward_prediction` contains four fields:

| Field | Meaning |
| --- | --- |
| `product_inchikey` | Stereo-free key of the expected product |
| `rank` | First matching position in the final distinct-product list, or `None` |
| `score` | `1 / rank`; `0.0` when no top-ten product matches |
| `alert` | Whether the expected product was absent from the top ten |

**One-reactant convention:** these steps receive `score=1.0`, `rank=None`, and
`alert=False` without inference. They were excluded from model training, so
this is a scoring convention, not a model prediction. Their score is displayed
in the same way as other positive scores.

The score measures **stereo-free product recovery**. It is not evidence of
stereochemical correctness or a probability of experimental success. Inference
errors remain errors rather than being converted into zero scores.

In [ ]:
import pandas as pd

reaction_table = pd.DataFrame(
    [
        {
            "route": route.provenance.tree_node_id,
            "step": step_index,
            "reaction": str(step.reaction),
            **step.forward_prediction,
        }
        for route in annotated_routes
        for step_index, step in enumerate(route.steps, start=1)
    ]
)
display(reaction_table)

In [ ]:
display(pd.Series(first_stats, name="First assessment"))

## 5. Aggregate route scores

`ForwardRouteScorer` computes the geometric mean across all $N$ steps:

$$F = \left(\prod_{i=1}^{N} s_i\right)^{1/N}.$$

Every step is counted once across all branches. A zero makes the route score
zero. One-reactant scores of 1.0 remain in the denominator, so `[0.25, 1.0]`
scores 0.5. Positive scores are aggregated using logarithms for numerical
stability.

Assessment returns copies in the input order and keeps the search score and
policy log likelihood separate. **Combined scoring is disabled by default.**
If explicitly needed, `ForwardRouteScorer(combine_search=True)` adds
`combined_score = search_score * forward_score`; no new predictions are needed.

In [ ]:
from synplan.chem.reaction.routes.quality import ForwardRouteScorer

scorer = ForwardRouteScorer()
scored_routes = scorer.assess(annotated_routes, silent=False)

route_table = pd.DataFrame(
    [
        {
            "route": route.provenance.tree_node_id,
            "steps": len(route),
            "forward_score": route.assessment["forward_score"],
            "search_score": route.assessment["search_score"],
            "zero_score_steps": route.assessment["zero_score_steps"],
        }
        for route in scored_routes
    ]
)
display(route_table)

Ranking is a separate, explicit operation. The following cell displays the
route with the highest forward score; `scored_routes` stays in its saved order.
Ties retain their input order. The drawing's step numbers correspond to the
reaction table above.

In [ ]:
ranked_routes = scorer.rank(scored_routes)
best_route = ranked_routes[0]
print(f"Route {best_route.provenance.tree_node_id}")
print(f"Forward route score: {best_route.assessment['forward_score']:.3f}")
display(best_route)

## 6. Export annotated JSON and an HTML report

Write a separate collection so the original planning output stays available.
The JSON retains reaction annotations under `meta.forward_prediction`, route
aggregates under `assessment`, and search provenance. Reading it back requires
neither the search tree nor a loaded forward model.

The self-contained HTML includes chemical drawings, reaction scores, and route
scores. Its filters show all routes, routes without zeros, or routes with at
least one zero. There is no combined-score column unless combination was enabled.

In [ ]:
from IPython.display import FileLink

from synplan.utils.visualisation import routes_report_html

annotated_path = results_dir / "lenvatinib_routes_forward.json"
html_path = results_dir / "lenvatinib_routes_forward.html"
write_routes_json(scored_routes, annotated_path)
routes_report_html(scored_routes, str(html_path), prices=False)

restored_routes = read_routes_json(annotated_path, as_routes=True)
print(f"Restored {len(restored_routes)} scored routes")
print("First saved assessment:", restored_routes[0].assessment)
display(FileLink(os.path.relpath(annotated_path)))
display(FileLink(os.path.relpath(html_path)))

## 7. Reopen the prediction cache

The named `forward_cache.json` is retained between runs. It stores ordered
products and their identities, likelihoods, and ranking scores. Its fingerprint
covers the model files, backend/runtime, generation settings, and preprocessing
policy; expected products are excluded from the input key.

A completed batch replaces the cache file atomically. After interruption,
completed inputs are reused and an unfinished batch may run again. Use separate
cache paths for concurrent processes.

Creating another assessor reloads the JSON cache while reusing the already
loaded model sessions. With identical settings, this second pass should show
**zero model predictions** and reproduce the same annotations and route scores.
Reassessment clears route aggregates, so run the route scorer again afterward.

In [ ]:
reopened = ForwardReactionAssessor(model, cache_path=cache_path, batch_size=1, top_k=10)
warm_annotations = reopened.assess(routes, silent=False)
warm_routes = scorer.assess(warm_annotations)
warm_stats = dict(reopened.last_stats)

print("New model predictions:", warm_stats["model_predictions"])
print(
    "Identical scored routes:",
    [route.to_json() for route in warm_routes]
    == [route.to_json() for route in scored_routes],
)
display(
    pd.DataFrame({"first assessment": first_stats, "reopened cache": warm_stats}).loc[
        [
            "reaction_occurrences",
            "unique_inputs",
            "cache_hits",
            "cache_misses",
            "model_predictions",
            "predictions_saved",
            "inference_seconds",
            "elapsed_seconds",
        ]
    ]
)

The first pass may already have cache hits if this notebook was run before.
`predictions_saved` counts avoided predictions relative to model-assessed
reaction occurrences; one-reactant exemptions are counted separately. Timing
comparisons should distinguish model loading, planning, and assessment.

To assess more of the saved routes, set `route_limit = None` and rerun from the
selection cell onward. Existing predictions remain reusable. For target-keyed
JSON/JSON.GZ planning collections, `scripts/assess_routes.py` provides the same
assessment and preserves existing collection metadata; see the
[forward-method documentation](https://synplanner.readthedocs.io/en/latest/methods/forward.html).